**Project 1**

**Setting up data:**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import (MinMaxScaler,MaxAbsScaler, RobustScaler)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression


features = [
    "matchId",
    "blueTeamControlWardsPlaced", "blueTeamWardsPlaced", "blueTeamTotalKills", "blueTeamDragonKills",
    "blueTeamHeraldKills", "blueTeamTowersDestroyed", "blueTeamInhibitorsDestroyed",
    "blueTeamTurretPlatesDestroyed", "blueTeamFirstBlood", "blueTeamMinionsKilled",
    "blueTeamJungleMinions", "blueTeamTotalGold", "blueTeamXp", "blueTeamTotalDamageToChamps",
    "redTeamControlWardsPlaced", "redTeamWardsPlaced", "redTeamTotalKills", "redTeamDragonKills",
    "redTeamHeraldKills", "redTeamTowersDestroyed", "redTeamInhibitorsDestroyed",
    "redTeamTurretPlatesDestroyed", "redTeamMinionsKilled", "redTeamJungleMinions",
    "redTeamTotalGold", "redTeamXp", "redTeamTotalDamageToChamps",
    "blueWin", "temp"
]

data = pd.read_csv('data.csv', names = features, na_values="?", skipinitialspace=True, skiprows=1)

data_without_classification = data.drop(columns=["matchId","blueWin", "temp"])
results_column = data["blueWin"]



X_train, X_test, y_train, y_test = train_test_split(
    data_without_classification, results_column,
    test_size=0.20,
    stratify=results_column,
    random_state=42,
)

print("train:", len(X_train))
print("test:", len(X_test))

train: 19380
test: 4845


**Benchmark we will compare our model with**

In [2]:
model = DecisionTreeClassifier(max_depth=6, random_state=42)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Model Test Accuracy: {accuracy * 100:.2f}%")
print(f"Train: {model.score(X_train, y_train) * 100:.2f}%")
print(f"Test:  {model.score(X_test, y_test) * 100:.2f}%")

Model Test Accuracy: 74.43%
Train: 76.36%
Test:  74.43%


**Model Engineering / K Nearest neighbours**

In [3]:
pipe = Pipeline([("scaling_methods", StandardScaler()), ("knn", KNeighborsClassifier())])

params = {
    "scaling_methods": [
        StandardScaler(),
        MinMaxScaler(),
        MaxAbsScaler(),
         RobustScaler(),
        "passthrough"
    ],
    "knn__n_neighbors": [3, 6, 8, 12, 18, 30, 40, 50,60,70,80, 90 , 150, 200],
    "knn__weights": ["uniform", "distance"]
}

grid = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best cross-validation accuracy:")
print(grid.best_score_)

Best parameters:
{'knn__n_neighbors': 200, 'knn__weights': 'distance', 'scaling_methods': StandardScaler()}
Best cross-validation accuracy:
0.7560887512899896


In [4]:
predictions = grid.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.7606


**Model Engineering / Logistic Regression**

In [5]:
clf = Pipeline([("scaler", StandardScaler()), ("logistic", LogisticRegression(max_iter=500, random_state=0))])
clf.fit(X_train, y_train)

predictions = clf.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy:.4f}")


Test accuracy: 0.7637


**Model Engineering / Decission Tree**

In [6]:

pipe = Pipeline([("dt", DecisionTreeClassifier())])

params = {
    "dt__criterion": ["gini", "entropy"],
    "dt__max_depth": [None, 3, 5, 10, 15, 20],
    "dt__min_samples_split": [2, 5, 10, 20],
    "dt__min_samples_leaf": [1, 2, 5, 10],
    "dt__max_features": [None, "sqrt", "log2"]
}

grid = GridSearchCV(
    pipe,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best cross-validation accuracy:")
print(grid.best_score_)

Best parameters:
{'dt__criterion': 'gini', 'dt__max_depth': 5, 'dt__max_features': None, 'dt__min_samples_leaf': 5, 'dt__min_samples_split': 2}
Best cross-validation accuracy:
0.7404024767801858


In [7]:
predictions = grid.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.7478


In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

model = HistGradientBoostingClassifier(random_state=42)

params = {
    "learning_rate": [0.03, 0.05, 0.1],
    "max_iter": [100, 200, 300],
    "max_leaf_nodes": [15, 31, 63],
    "l2_regularization": [0, 0.1, 1, 5]
}

grid = GridSearchCV(
    model,
    params,
    cv=5,
    scoring="accuracy",
    n_jobs=2
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best cross-validation accuracy:")
print(grid.best_score_)

predictions = grid.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy:.4f}")

Best parameters:
{'l2_regularization': 1, 'learning_rate': 0.03, 'max_iter': 200, 'max_leaf_nodes': 15}
Best cross-validation accuracy:
0.758513931888545
Test accuracy: 0.7635
